In [1]:
import os, random
import numpy as np
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# InsightFace + onnxruntime
try:
    import insightface
    from insightface.app import FaceAnalysis
except Exception:
    !pip -q uninstall -y onnxruntime
    !pip -q install insightface onnxruntime-gpu
    import insightface
    from insightface.app import FaceAnalysis

print("insightface:", insightface.__version__)


insightface: 0.7.3


In [2]:
import onnxruntime as ort
print(ort.get_available_providers())

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [3]:
import cv2

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

def get_largest_face(faces):
    if not faces:
        return None
    areas = []
    for f in faces:
        x1, y1, x2, y2 = f.bbox.astype(int)
        areas.append((x2-x1) * (y2-y1))
    return faces[int(np.argmax(areas))]

def image_to_embedding(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    faces = app.get(img)
    face = get_largest_face(faces)
    if face is None:
        return None
    emb = face.embedding.astype(np.float32)
    emb /= (np.linalg.norm(emb) + 1e-12)
    return emb


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

In [4]:
import os
import kagglehub

# Download dataset
LFW_ROOT = kagglehub.dataset_download("jessicali9530/lfw-dataset")

LFW_ROOT = os.path.join(LFW_ROOT, "lfw-deepfunneled", "lfw-deepfunneled")

if not os.path.isdir(LFW_ROOT):
    raise FileNotFoundError(f"LFW_ROOT not found: {LFW_ROOT}")

identities = [d for d in os.listdir(LFW_ROOT) if os.path.isdir(os.path.join(LFW_ROOT, d))]

print("LFW_ROOT:", LFW_ROOT)
print("Number of identities:", len(identities))
print("Sample identities:", identities[:10])

Using Colab cache for faster access to the 'lfw-dataset' dataset.
LFW_ROOT: /kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled
Number of identities: 5749
Sample identities: ['Tyler_Hamilton', 'Bernard_Siegel', 'Blythe_Danner', 'Gene_Robinson', 'Nicole_Parker', 'Coco_dEste', 'Bernard_Ebbers', 'Ralph_Sampson', 'Adam_Herbert', 'Colin_Powell']


In [5]:
def list_identities(root):
    ids = []
    for name in os.listdir(root):
        d = os.path.join(root, name)
        if not os.path.isdir(d):
            continue
        imgs = [
            os.path.join(d, f)
            for f in os.listdir(d)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]
        if len(imgs) >= 2:
            ids.append((name, sorted(imgs)))
    return ids

identities = list_identities(LFW_ROOT)
print("identities (>=2 imgs):", len(identities))

rng = np.random.default_rng(SEED)
rng.shuffle(identities)

n = len(identities)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

train_ids = identities[:n_train]
val_ids   = identities[n_train:n_train+n_val]
test_ids  = identities[n_train+n_val:]

print("train/val/test:", len(train_ids), len(val_ids), len(test_ids))


identities (>=2 imgs): 1680
train/val/test: 1176 252 252


In [6]:
def build_gallery(id_list, ref_index=0, max_ids=None):
    gallery = {}
    used = id_list if max_ids is None else id_list[:max_ids]
    skipped = 0

    for person, imgs in tqdm(used, desc="gallery"):
        ref_img = imgs[ref_index]
        emb = image_to_embedding(ref_img)
        if emb is None:
            skipped += 1
            continue
        gallery[person] = {"ref_img": ref_img, "ref_emb": emb}
    return gallery, skipped

MAX_TRAIN_IDS = 1200
MAX_VAL_IDS   = 400
MAX_TEST_IDS  = 400

train_gallery, s1 = build_gallery(train_ids, max_ids=MAX_TRAIN_IDS)
val_gallery,   s2 = build_gallery(val_ids,   max_ids=MAX_VAL_IDS)
test_gallery,  s3 = build_gallery(test_ids,  max_ids=MAX_TEST_IDS)

print("train_gallery:", len(train_gallery), "skipped:", s1)
print("val_gallery:", len(val_gallery), "skipped:", s2)
print("test_gallery:", len(test_gallery), "skipped:", s3)


gallery:   0%|          | 0/1176 [00:00<?, ?it/s]

gallery:   0%|          | 0/252 [00:00<?, ?it/s]

gallery:   0%|          | 0/252 [00:00<?, ?it/s]

train_gallery: 1172 skipped: 4
val_gallery: 251 skipped: 1
test_gallery: 252 skipped: 0


In [7]:
def build_probes(id_list, gallery, ref_index=0, max_probes_per_id=3):
    probes = []
    rng = np.random.default_rng(SEED)
    for person, imgs in id_list:
        if person not in gallery:
            continue
        other_imgs = [p for i, p in enumerate(imgs) if i != ref_index]
        rng.shuffle(other_imgs)
        for p in other_imgs[:max_probes_per_id]:
            probes.append((person, p))
    return probes

val_probes  = build_probes(val_ids[:MAX_VAL_IDS],  val_gallery,  max_probes_per_id=3)
test_probes = build_probes(test_ids[:MAX_TEST_IDS], test_gallery, max_probes_per_id=3)

print("val probes:", len(val_probes))
print("test probes:", len(test_probes))


val probes: 489
test probes: 479


In [8]:
def cosine_sim(a, b):
    return float(np.dot(a, b))  # both are L2-normalized

def best_match(probe_emb, gallery):
    best_person = None
    best_score = -1.0
    for person, data in gallery.items():
        score = cosine_sim(probe_emb, data["ref_emb"])
        if score > best_score:
            best_score = score
            best_person = person
    return best_person, best_score

def run_probes(probes, gallery):
    y_true, y_pred, scores = [], [], []
    skipped = 0
    for true_person, img_path in tqdm(probes, desc="probes"):
        emb = image_to_embedding(img_path)
        if emb is None:
            skipped += 1
            continue
        pred_person, score = best_match(emb, gallery)
        y_true.append(true_person)
        y_pred.append(pred_person)
        scores.append(score)
    return np.array(y_true), np.array(y_pred), np.array(scores, np.float32), skipped


In [9]:
val_y_true, val_y_pred, val_scores, val_skipped = run_probes(val_probes, val_gallery)
print("val evaluated:", len(val_scores), "skipped:", val_skipped)

val_is_correct = (val_y_true == val_y_pred).astype(np.int32)

def metrics_at_threshold(is_correct, scores, T):
    accept = (scores >= T).astype(np.int32)

    TP = int(np.sum((is_correct == 1) & (accept == 1)))
    FN = int(np.sum((is_correct == 1) & (accept == 0)))
    FP = int(np.sum((is_correct == 0) & (accept == 1)))
    TN = int(np.sum((is_correct == 0) & (accept == 0)))

    precision = TP / (TP + FP + 1e-12)
    recall    = TP / (TP + FN + 1e-12)
    f1        = 2 * precision * recall / (precision + recall + 1e-12)

    FAR = FP / (FP + TN + 1e-12)
    FRR = FN / (FN + TP + 1e-12)

    return {"T": float(T), "precision": precision, "recall": recall, "f1": f1, "FAR": FAR, "FRR": FRR,
            "TP": TP, "FP": FP, "TN": TN, "FN": FN}

Ts = np.linspace(0.2, 0.9, 200)

best = None
for T in Ts:
    m = metrics_at_threshold(val_is_correct, val_scores, T)
    if m["FAR"] <= 0.01:
        if best is None or m["f1"] > best["f1"]:
            best = m

if best is None:
    best = max((metrics_at_threshold(val_is_correct, val_scores, T) for T in Ts), key=lambda x: x["f1"])

best


probes:   0%|          | 0/489 [00:00<?, ?it/s]

val evaluated: 486 skipped: 3


{'T': 0.5095477386934673,
 'precision': 0.9999999999999977,
 'recall': 0.9339019189765438,
 'f1': 0.9658213891946471,
 'FAR': 0.0,
 'FRR': 0.06609808102345402,
 'TP': 438,
 'FP': 0,
 'TN': 17,
 'FN': 31}

In [10]:
TEST_T = best["T"]
print("chosen threshold:", TEST_T)

test_y_true, test_y_pred, test_scores, test_skipped = run_probes(test_probes, test_gallery)
print("test evaluated:", len(test_scores), "skipped:", test_skipped)

test_is_correct = (test_y_true == test_y_pred).astype(np.int32)
test_metrics = metrics_at_threshold(test_is_correct, test_scores, TEST_T)
test_metrics


chosen threshold: 0.5095477386934673


probes:   0%|          | 0/479 [00:00<?, ?it/s]

test evaluated: 479 skipped: 0


{'T': 0.5095477386934673,
 'precision': 0.9999999999999977,
 'recall': 0.9397849462365571,
 'f1': 0.9689578713963939,
 'FAR': 0.0,
 'FRR': 0.060215053763440725,
 'TP': 437,
 'FP': 0,
 'TN': 14,
 'FN': 28}

In [11]:
# Accuracy metrics

# 1) Top-1 identification accuracy (closed-set): how often best_match label is correct (no threshold).
top1_acc_val  = float(np.mean(val_is_correct)) if len(val_is_correct) else float("nan")
top1_acc_test = float(np.mean(test_is_correct)) if len(test_is_correct) else float("nan")

# 2) Access-decision accuracy (thresholded): (TP + TN) / total, where negatives are "wrong match attempts".
TP, FP, TN, FN = test_metrics["TP"], test_metrics["FP"], test_metrics["TN"], test_metrics["FN"]
access_acc_test = (TP + TN) / (TP + TN + FP + FN + 1e-12)

print(f"Top-1 Identification Accuracy (val):  {top1_acc_val*100:.2f}%")
print(f"Top-1 Identification Accuracy (test): {top1_acc_test*100:.2f}%")
print(f"Access-Decision Accuracy at T={TEST_T:.4f} (test): {access_acc_test*100:.2f}%")

# Keep the full metrics dict visible too
test_metrics


Top-1 Identification Accuracy (val):  96.50%
Top-1 Identification Accuracy (test): 97.08%
Access-Decision Accuracy at T=0.5095 (test): 94.15%


{'T': 0.5095477386934673,
 'precision': 0.9999999999999977,
 'recall': 0.9397849462365571,
 'f1': 0.9689578713963939,
 'FAR': 0.0,
 'FRR': 0.060215053763440725,
 'TP': 437,
 'FP': 0,
 'TN': 14,
 'FN': 28}

In [12]:
# Open-set evaluation

def build_impostor_probes(all_ids, gallery, max_impostor_ids=300, max_imgs_per_id=2):
    gallery_names = set(gallery.keys())
    impostor_ids = [(name, imgs) for (name, imgs) in all_ids if name not in gallery_names]
    rng = np.random.default_rng(SEED)
    rng.shuffle(impostor_ids)
    impostor_ids = impostor_ids[:max_impostor_ids]

    probes = []
    for name, imgs in impostor_ids:
        # take a few images per impostor identity
        rng.shuffle(imgs)
        for p in imgs[:max_imgs_per_id]:
            probes.append((name, p))
    return probes

# Build a pool of all identities (train+val+test) and then take impostors not in test_gallery.
all_ids_pool = train_ids + val_ids + test_ids
impostor_probes = build_impostor_probes(all_ids_pool, test_gallery, max_impostor_ids=300, max_imgs_per_id=2)

imp_y_true, imp_y_pred, imp_scores, imp_skipped = run_probes(impostor_probes, test_gallery)
print("impostor evaluated:", len(imp_scores), "skipped:", imp_skipped)

# Open-set FAR: fraction of impostor attempts that get accepted (score >= T)
imp_accept = (imp_scores >= TEST_T).astype(np.int32)
open_set_FAR = float(np.mean(imp_accept)) if len(imp_accept) else float("nan")

# Open-set TAR (true accept rate for authorized users) measured on only correct matches
genuine_correct_scores = test_scores[test_is_correct == 1]
genuine_accept = (genuine_correct_scores >= TEST_T).astype(np.int32)
open_set_TAR = float(np.mean(genuine_accept)) if len(genuine_accept) else float("nan")
open_set_FRR = 1.0 - open_set_TAR if not np.isnan(open_set_TAR) else float("nan")

print(f"Open-set FAR (unknown accepted) at T={TEST_T:.4f}: {open_set_FAR*100:.3f}%")
print(f"Open-set TAR (authorized accepted) at T={TEST_T:.4f}: {open_set_TAR*100:.2f}%")
print(f"Open-set FRR (authorized rejected) at T={TEST_T:.4f}: {open_set_FRR*100:.2f}%")


probes:   0%|          | 0/600 [00:00<?, ?it/s]

impostor evaluated: 597 skipped: 3
Open-set FAR (unknown accepted) at T=0.5095: 0.000%
Open-set TAR (authorized accepted) at T=0.5095: 93.98%
Open-set FRR (authorized rejected) at T=0.5095: 6.02%


In [13]:
# ----------------------------
#Deployment helpers
# ----------------------------

def pack_gallery(gallery):
    """Convert gallery dict -> (names list, embedding matrix [N, D])"""
    names = list(gallery.keys())
    embs = np.stack([gallery[n]["ref_emb"] for n in names], axis=0).astype(np.float32)

    embs /= (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12)
    return names, embs

def best_match_vectorized(probe_emb, names, emb_matrix):
    """Return best match using one matrix multiply (cosine similarity because both are normalized)."""
    scores = emb_matrix @ probe_emb.astype(np.float32)
    j = int(np.argmax(scores))
    return names[j], float(scores[j])

# Example: create packed gallery for the test gallery
packed_names, packed_embs = pack_gallery(test_gallery)

# Quick sanity check that vectorized and loop versions agree on a few probes
for k in range(min(5, len(test_probes))):
    _, img_path = test_probes[k]
    emb = image_to_embedding(img_path)
    if emb is None:
        continue
    p1, s1 = best_match(emb, test_gallery)
    p2, s2 = best_match_vectorized(emb, packed_names, packed_embs)
    if p1 != p2:
        print("WARNING mismatch:", p1, p2, s1, s2)
        break
print("Packed gallery ready. Size:", len(packed_names))

# Save artifacts for deployment (gallery + threshold)
ARTIFACT_DIR = "fr_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

np.save(os.path.join(ARTIFACT_DIR, "gallery_embs.npy"), packed_embs)
with open(os.path.join(ARTIFACT_DIR, "gallery_names.txt"), "w") as f:
    for n in packed_names:
        f.write(n + "\n")
with open(os.path.join(ARTIFACT_DIR, "threshold.txt"), "w") as f:
    f.write(str(TEST_T))

print("Saved:", os.listdir(ARTIFACT_DIR))


Packed gallery ready. Size: 252
Saved: ['threshold.txt', 'gallery_names.txt', 'gallery_embs.npy']


In [14]:
def check_access(img_path, gallery, T):
    emb = image_to_embedding(img_path)
    if emb is None:
        return {"status": "no_face", "authorized": False, "match": None, "score": None}

    person, score = best_match(emb, gallery)
    authorized = (score >= T)
    return {"status": "ok", "authorized": bool(authorized), "match": person, "score": float(score)}

# Demo
if len(test_probes) > 0:
    true_person, probe_img = test_probes[0]
    out = check_access(probe_img, test_gallery, TEST_T)
    print("true:", true_person)
    print(out)


true: Nicole_Kidman
{'status': 'ok', 'authorized': True, 'match': 'Nicole_Kidman', 'score': 0.7034735679626465}


In [15]:
import os
import numpy as np

# -------- choose identity --------
PERSON_NAME = "George_W_Bush"
person_dir = os.path.join(LFW_ROOT, PERSON_NAME)

if not os.path.isdir(person_dir):
    raise FileNotFoundError(f"Person folder not found: {person_dir}")

person_imgs = sorted([
    os.path.join(person_dir, f)
    for f in os.listdir(person_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print("Person:", PERSON_NAME)
print("Total images found:", len(person_imgs))
print("First few images:")
for p in person_imgs[:5]:
    print(" -", p)

# use first 10 images for gallery
GALLERY_COUNT = 10
gallery_imgs = person_imgs[:GALLERY_COUNT]

print("\nUsing these images for enrollment/gallery:")
for p in gallery_imgs:
    print(" -", os.path.basename(p))

Person: George_W_Bush
Total images found: 530
First few images:
 - /kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0001.jpg
 - /kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0002.jpg
 - /kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0003.jpg
 - /kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0004.jpg
 - /kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled/George_W_Bush/George_W_Bush_0005.jpg

Using these images for enrollment/gallery:
 - George_W_Bush_0001.jpg
 - George_W_Bush_0002.jpg
 - George_W_Bush_0003.jpg
 - George_W_Bush_0004.jpg
 - George_W_Bush_0005.jpg
 - George_W_Bush_0006.jpg
 - George_W_Bush_0007.jpg
 - George_W_Bush_0008.jpg
 - George_W_Bush_0009.jpg
 - George_W_Bush_0010.jpg


In [16]:
# Build a stronger gallery embedding by averaging multiple George images
embs = []
bad_imgs = []

for img_path in gallery_imgs:
    emb = image_to_embedding(img_path)
    if emb is None:
        bad_imgs.append(img_path)
    else:
        embs.append(emb)

if len(embs) == 0:
    raise RuntimeError("No valid embeddings were created for George_W_Bush.")

embs = np.stack(embs, axis=0).astype(np.float32)
george_emb = np.mean(embs, axis=0)
george_emb /= (np.linalg.norm(george_emb) + 1e-12)

george_gallery = {
    PERSON_NAME: {
        "ref_img": gallery_imgs[0],
        "ref_emb": george_emb
    }
}

print("Valid gallery embeddings used:", len(embs))
print("Failed gallery images:", len(bad_imgs))
if bad_imgs:
    print("Failed files:")
    for p in bad_imgs:
        print(" -", p)

Valid gallery embeddings used: 10
Failed gallery images: 0


In [17]:
# Pack George-only gallery
george_names = [PERSON_NAME]
george_embs = np.stack([george_gallery[PERSON_NAME]["ref_emb"]], axis=0).astype(np.float32)
george_embs /= (np.linalg.norm(george_embs, axis=1, keepdims=True) + 1e-12)

# Keep the threshold from your original notebook if it exists
try:
    GEORGE_T = float(TEST_T)
except NameError:
    GEORGE_T = 0.5095477386934673  # fallback from your notebook

print("George gallery size:", len(george_names))
print("Threshold used:", GEORGE_T)

George gallery size: 1
Threshold used: 0.5095477386934673


In [18]:
import cv2

def cosine_sim(a, b):
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    a /= (np.linalg.norm(a) + 1e-12)
    b /= (np.linalg.norm(b) + 1e-12)
    return float(np.dot(a, b))

def best_match_vectorized(probe_emb, names, emb_matrix):
    probe_emb = probe_emb.astype(np.float32)
    probe_emb /= (np.linalg.norm(probe_emb) + 1e-12)
    scores = emb_matrix @ probe_emb
    idx = int(np.argmax(scores))
    return names[idx], float(scores[idx])

def annotate_frame(frame, face, label, score, authorized):
    x1, y1, x2, y2 = face.bbox.astype(int)
    color = (0, 255, 0) if authorized else (0, 0, 255)

    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    text = f"{label} | {score:.3f}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)

    y_text_top = max(0, y1 - th - 10)
    cv2.rectangle(frame, (x1, y_text_top), (x1 + tw + 8, y1), color, -1)
    cv2.putText(
        frame,
        text,
        (x1 + 4, y1 - 6),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )
    return frame

In [19]:
# -------- set your video path here --------
INPUT_VIDEO = "/content/George W Bush interview.mp4"
OUTPUT_VIDEO = "george_bush_result.mp4"

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

frame_idx = 0
processed_frames = 0
recognized_frames = 0
unknown_frames = 0
no_face_frames = 0
all_scores = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    processed_frames += 1

    faces = app.get(frame)

    if len(faces) == 0:
        no_face_frames += 1
        writer.write(frame)
        continue

    # If multiple faces appear, take the largest one
    face = get_largest_face(faces)

    emb = face.embedding.astype(np.float32)
    emb /= (np.linalg.norm(emb) + 1e-12)

    pred_name, score = best_match_vectorized(emb, george_names, george_embs)
    all_scores.append(score)

    authorized = score >= GEORGE_T

    if authorized:
        recognized_frames += 1
        shown_label = pred_name
    else:
        unknown_frames += 1
        shown_label = "Unknown"

    frame = annotate_frame(frame, face, shown_label, score, authorized)
    writer.write(frame)

cap.release()
writer.release()

print("Finished.")
print("Saved output video:", OUTPUT_VIDEO)
print("Processed frames:", processed_frames)
print("Frames with no face:", no_face_frames)
print("Recognized frames:", recognized_frames)
print("Unknown frames:", unknown_frames)

if all_scores:
    print("Min score:", float(np.min(all_scores)))
    print("Mean score:", float(np.mean(all_scores)))
    print("Max score:", float(np.max(all_scores)))
else:
    print("No scores were produced.")

Finished.
Saved output video: george_bush_result.mp4
Processed frames: 731
Frames with no face: 0
Recognized frames: 731
Unknown frames: 0
Min score: 0.6748065948486328
Mean score: 0.7412813808131968
Max score: 0.7858704328536987


In [20]:
if all_scores:
    accept_rate = recognized_frames / max(1, (recognized_frames + unknown_frames))
    print(f"Acceptance rate over detected-face frames: {accept_rate*100:.2f}%")

    if np.mean(all_scores) >= GEORGE_T:
        print("Overall: the video tends to match George_W_Bush.")
    else:
        print("Overall: the video does not match strongly enough with the current threshold.")
else:
    print("No face embeddings were extracted from the video.")

Acceptance rate over detected-face frames: 100.00%
Overall: the video tends to match George_W_Bush.


In [21]:
# Re-run quickly and print one score every 10 frames
INPUT_VIDEO = "/content/George W Bush interview.mp4"

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

frame_idx = 0
sampled = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    if frame_idx % 10 != 0:
        continue

    faces = app.get(frame)
    if len(faces) == 0:
        sampled.append((frame_idx, None))
        continue

    face = get_largest_face(faces)
    emb = face.embedding.astype(np.float32)
    emb /= (np.linalg.norm(emb) + 1e-12)

    pred_name, score = best_match_vectorized(emb, george_names, george_embs)
    sampled.append((frame_idx, score))

cap.release()

print("Sampled frame scores:")
for item in sampled[:20]:
    print(item)

Sampled frame scores:
(10, 0.7601146697998047)
(20, 0.7594876885414124)
(30, 0.7663907408714294)
(40, 0.779200553894043)
(50, 0.7640478014945984)
(60, 0.7658060789108276)
(70, 0.760330080986023)
(80, 0.7614506483078003)
(90, 0.7692155241966248)
(100, 0.7379727363586426)
(110, 0.7712171077728271)
(120, 0.7747112512588501)
(130, 0.7383682727813721)
(140, 0.749393880367279)
(150, 0.7594940662384033)
(160, 0.7534526586532593)
(170, 0.7597978115081787)
(180, 0.7734580039978027)
(190, 0.7335802316665649)
(200, 0.7494431734085083)


In [25]:
import os
import numpy as np
import cv2

# ==============================
# SETTINGS
# ==============================
person_name = "George_W_Bush"
person_path = os.path.join(LFW_ROOT, person_name)

gallery_embeddings = []

# ==============================
# BUILD GALLERY
# ==============================
for img_name in os.listdir(person_path):
    img_path = os.path.join(person_path, img_name)

    img = cv2.imread(img_path)
    if img is None:
        continue

    faces = app.get(img)

    if len(faces) == 0:
        continue

    emb = faces[0].embedding
    emb = emb / np.linalg.norm(emb)

    gallery_embeddings.append(emb)

print("Gallery size:", len(gallery_embeddings))

Gallery size: 529


In [26]:
import cv2
import numpy as np

# ==============================
# SETTINGS
# ==============================
video_path = "/content/YTDown.com_YouTube_Rep-James-Talarico-On-Confronting-Christ_Media_oiTJ7Pz_59A_001_1080p - Trim.mp4"
output_path = "unknown_result.mp4"

THRESHOLD = 0.5095   # same threshold you used


gallery = np.vstack(gallery_embeddings)

# ==============================
# VIDEO PROCESSING
# ==============================
cap = cv2.VideoCapture(video_path)

fps = int(cap.get(cv2.CAP_PROP_FPS))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (w, h)
)

frame_count = 0
unknown_count = 0
recognized_count = 0
scores = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # detect + embed using your existing InsightFace model
    faces = app.get(frame)

    if len(faces) == 0:
        label = "No Face"
    else:
        emb = faces[0].embedding
        emb = emb / np.linalg.norm(emb)

        # cosine similarity with gallery
        sims = gallery @ emb
        max_score = np.max(sims)

        scores.append(max_score)

        if max_score >= THRESHOLD:
            label = f"George ({max_score:.3f})"
            recognized_count += 1
        else:
            label = f"Unknown ({max_score:.3f})"
            unknown_count += 1

    # draw label
    cv2.putText(frame, label, (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                (0, 0, 255), 2)

    out.write(frame)

cap.release()
out.release()

# ==============================
# RESULTS
# ==============================
print("Finished.")
print(f"Saved output video: {output_path}")
print(f"Processed frames: {frame_count}")
print(f"Recognized frames: {recognized_count}")
print(f"Unknown frames: {unknown_count}")

if len(scores) > 0:
    print(f"Min score: {np.min(scores)}")
    print(f"Mean score: {np.mean(scores)}")
    print(f"Max score: {np.max(scores)}")
else:
    print("No scores were produced.")

Finished.
Saved output video: unknown_result.mp4
Processed frames: 493
Recognized frames: 0
Unknown frames: 493
Min score: 0.03481739014387131
Mean score: 0.10590877383947372
Max score: 0.16583657264709473
